# Feature Engineering Combination v5 — tujuh kelompok fitur


## 1. Konfigurasi dan lokasi artefak v5


In [ ]:
from pathlib import Path
import hashlib, json, os, time
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 30092026
RUN_ID = "cifar10_7fitur_v5"
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / "fusion"
for folder in ("model", "fitur", "laporan"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)
VARIANTS = ("rgb", "avg", "ntsc")
BLOCK_NAMES = ("rgb", "avg", "ntsc", "hog", "lbp", "hsv", "rgb_stats")
BOUNDARIES = (0, 512, 1024, 1536, 1860, 2116, 2164, 2173)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


print("Output fusion:", OUT)


## 2. Muat official training dan split tetap


In [ ]:
from tensorflow.keras.datasets import cifar10

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
idx_train, idx_val = train_test_split(
    np.arange(50000), test_size=0.2, random_state=SEED, stratify=y_all
)
idx_train, idx_val = np.sort(idx_train), np.sort(idx_val)
split_hash = hashlib.sha256(idx_train.tobytes() + idx_val.tobytes()).hexdigest()
if x_all.shape != (50000, 32, 32, 3):
    raise ValueError("Official training CIFAR-10 tidak sesuai")
print("Split:", split_hash)


## 3. Validasi dan muat tiga arsip embedding CNN


In [ ]:
embeddings = {}
model_hashes = {}
archive_hashes = {}
model_paths = {}
for variant in VARIANTS:
    base = ROOT / "outputs" / RUN_ID / variant
    model_path = base / "model" / "cnn.keras"
    feature_path = base / "fitur" / "train_features.npz"
    report_path = base / "laporan" / "cnn_training.json"
    for path in (model_path, feature_path, report_path):
        if not path.is_file():
            raise FileNotFoundError(str(path))
    report = json.loads(report_path.read_text(encoding="utf-8"))
    model_hash = sha256_file(model_path)
    if report["cnn_sha256"] != model_hash or report["split_hash"] != split_hash:
        raise ValueError("Checkpoint tidak cocok: " + variant)
    with np.load(feature_path, allow_pickle=False) as archive:
        if (
            str(archive["run_id"]) != RUN_ID
            or str(archive["variant"]) != variant
            or str(archive["cnn_sha256"]) != model_hash
            or str(archive["split_hash"]) != split_hash
            or str(archive["feature_layer"]) != "embedding"
            or str(archive["preprocessing_version"])
            != "v5-float32-div255-original-flip"
            or not np.array_equal(archive["indices"], np.arange(50000))
            or not np.array_equal(archive["y"], y_all)
            or not np.array_equal(archive["train_idx"], idx_train)
            or not np.array_equal(archive["val_idx"], idx_val)
        ):
            raise ValueError("Metadata fitur tidak cocok: " + variant)
        a = archive["features_A"].astype(np.float32)
        b = archive["features_B"].astype(np.float32)
    if a.shape != (50000, 512) or b.shape != (50000, 512):
        raise ValueError("Dimensi embedding salah: " + variant)
    if not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError("Embedding tidak berhingga: " + variant)
    embeddings[variant] = {"A": a, "B": b}
    model_hashes[variant] = model_hash
    archive_hashes[variant] = sha256_file(feature_path)
    model_paths[variant] = str(model_path.relative_to(ROOT))
    print("Fitur cocok:", variant, a.shape, "SHA", archive_hashes[variant][:16])


## 4. Periksa kemungkinan penggunaan ulang descriptor v2


In [ ]:
V2 = ROOT / "outputs" / "cifar10_7fitur_v2" / "fusion"
old_manifest = V2 / "laporan" / "combination_manifest.json"
old_archive = V2 / "fitur" / "train_combined_B.npz"
manual = None
manual_source = None
if old_manifest.exists() and old_archive.exists():
    old = json.loads(old_manifest.read_text(encoding="utf-8"))
    if (
        old.get("split_hash") == split_hash
        and old.get("archive_sha256", {}).get("B") == sha256_file(old_archive)
        and old.get("boundaries", {}).get("B") == list(BOUNDARIES)
    ):
        with np.load(old_archive, allow_pickle=False) as data:
            if (
                str(data["run_id"]) == "cifar10_7fitur_v2"
                and str(data["representation"]) == "B"
                and str(data["split_hash"]) == split_hash
                and str(data["preprocessing_version"]) == old["preprocessing_version"]
                and data["features"].shape == (50000, 2173)
                and np.array_equal(data["indices"], np.arange(50000))
                and np.array_equal(data["y"], y_all)
                and np.array_equal(data["train_idx"], idx_train)
                and np.array_equal(data["val_idx"], idx_val)
                and np.array_equal(data["boundaries"], BOUNDARIES)
                and np.array_equal(data["model_hashes"], old["model_hashes"])
            ):
                manual = data["features"][:, 1536:].astype(np.float32)
                manual_source = {
                    "kind": "verified_v2_archive",
                    "sha256": sha256_file(old_archive),
                    "manifest_sha256": sha256_file(old_manifest),
                }
if manual is not None:
    if manual.shape != (50000, 637) or not np.isfinite(manual).all():
        raise ValueError("Descriptor v2 invalid")
    print("Empat descriptor manual digunakan kembali dari arsip v2 terverifikasi")
else:
    print("Arsip v2 tidak cocok/tersedia; descriptor akan diekstrak ulang")


## 5. Definisi empat descriptor manual


In [ ]:
def handcrafted_descriptors(x, batch_size=128, verbose=1):
    # HOG: 9 orientasi unsigned, cell 8x8, block 2x2, L2 per block.
    # LBP: 8 tetangga radius 1, histogram 256 kode.
    # HSV: 16 bin per kanal. Statistik RGB: mean, std, skewness per kanal.
    import matplotlib.colors as mcolors

    outputs = {k: [] for k in ("hog", "lbp", "hsv", "rgb_stats")}
    total_batches = int(np.ceil(len(x) / batch_size))
    progress_interval = max(1, total_batches // 20)
    for begin in range(0, len(x), batch_size):
        rgb = np.asarray(x[begin : begin + batch_size], dtype=np.float32) / 255.0
        gray = np.sum(rgb * np.array([0.299, 0.587, 0.114], np.float32), axis=-1)
        gy, gx = np.gradient(gray, axis=(1, 2))
        magnitude = np.sqrt(gx * gx + gy * gy)
        orientation = np.mod(np.arctan2(gy, gx) * (180.0 / np.pi), 180.0)
        bins = np.minimum((orientation / 20.0).astype(np.int32), 8)
        cell_hist = np.zeros((len(rgb), 4, 4, 9), dtype=np.float32)
        for row in range(4):
            for col in range(4):
                b = bins[:, row * 8 : (row + 1) * 8, col * 8 : (col + 1) * 8]
                m = magnitude[:, row * 8 : (row + 1) * 8, col * 8 : (col + 1) * 8]
                for angle in range(9):
                    cell_hist[:, row, col, angle] = np.sum(
                        m * (b == angle), axis=(1, 2)
                    )
        blocks = []
        for row in range(3):
            for col in range(3):
                block = cell_hist[:, row : row + 2, col : col + 2, :].reshape(
                    len(rgb), -1
                )
                denominator = np.maximum(
                    np.linalg.norm(block, axis=1, keepdims=True), 1e-8
                )
                blocks.append(block / denominator)
        outputs["hog"].append(np.concatenate(blocks, axis=1))
        padded = np.pad(gray, ((0, 0), (1, 1), (1, 1)), mode="edge")
        lbp = np.zeros(gray.shape, dtype=np.uint8)
        offsets = [(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]
        for bit, (dy, dx) in enumerate(offsets):
            neighbor = padded[:, 1 + dy : 33 + dy, 1 + dx : 33 + dx]
            lbp |= (neighbor >= gray).astype(np.uint8) << bit
        hist = np.zeros((len(rgb), 256), dtype=np.float32)
        np.add.at(hist, (np.repeat(np.arange(len(rgb)), 1024), lbp.reshape(-1)), 1)
        outputs["lbp"].append(hist / 1024.0)
        hsv = mcolors.rgb_to_hsv(rgb)
        hsv_hist = np.zeros((len(rgb), 3, 16), dtype=np.float32)
        for channel in range(3):
            hsv_bin = np.minimum((hsv[..., channel] * 16).astype(np.int32), 15)
            np.add.at(
                hsv_hist[:, channel, :],
                (np.repeat(np.arange(len(rgb)), 1024), hsv_bin.reshape(-1)),
                1,
            )
        outputs["hsv"].append(hsv_hist.reshape(len(rgb), -1) / 1024.0)
        flat = rgb.reshape(len(rgb), -1, 3)
        mean = flat.mean(axis=1)
        std = flat.std(axis=1)
        skew = np.mean(
            ((flat - mean[:, None, :]) / np.maximum(std[:, None, :], 1e-6)) ** 3, axis=1
        )
        outputs["rgb_stats"].append(np.concatenate((mean, std, skew), axis=1))
        batch_number = begin // batch_size + 1
        if verbose == 1 and (
            batch_number % progress_interval == 0 or batch_number == total_batches
        ):
            processed = min(begin + batch_size, len(x))
            print(f"Descriptor: {processed}/{len(x)} gambar", flush=True)
    result = {
        key: np.concatenate(parts).astype(np.float32) for key, parts in outputs.items()
    }
    assert [result[k].shape[1] for k in ("hog", "lbp", "hsv", "rgb_stats")] == [
        324,
        256,
        48,
        9,
    ]
    assert all(np.isfinite(value).all() for value in result.values())
    return result


## 6. Ekstraksi descriptor bila cache tidak valid


In [ ]:
if manual is None:
    began = time.perf_counter()
    desc = handcrafted_descriptors(x_all, batch_size=128, verbose=1)
    manual = np.concatenate(
        [desc[k] for k in ("hog", "lbp", "hsv", "rgb_stats")], axis=1
    ).astype(np.float32)
    manual_source = {"kind": "recomputed_v5", "recipe": "v4-handcrafted-compatible"}
    print("Descriptor dihitung ulang, detik:", round(time.perf_counter() - began, 1))
if manual.shape != (50000, 637) or not np.isfinite(manual).all():
    raise ValueError("Empat descriptor manual harus 637 dimensi dan finite")
print("Descriptor manual:", manual.shape, manual_source)


## 7. Gabungkan tujuh kelompok fitur A dan B


In [ ]:
combined = {}
for rep in ("A", "B"):
    combined[rep] = np.concatenate(
        [embeddings[v][rep] for v in VARIANTS] + [manual], axis=1
    ).astype(np.float32)
    if combined[rep].shape != (50000, 2173) or not np.isfinite(combined[rep]).all():
        raise ValueError("Fusion " + rep + " invalid")
    print("Representasi:", rep, combined[rep].shape)


## 8. Simpan arsip kombinasi A dan B


In [ ]:
combined_hashes = {}
for rep, matrix in combined.items():
    path = OUT / "fitur" / f"train_combined_{rep}.npz"
    if path.exists():
        with np.load(path, allow_pickle=False) as old:
            if (
                str(old["run_id"]) != RUN_ID
                or str(old["split_hash"]) != split_hash
                or not np.array_equal(
                    old["model_hashes"], [model_hashes[v] for v in VARIANTS]
                )
                or not np.array_equal(old["indices"], np.arange(50000))
                or not np.array_equal(old["y"], y_all)
                or not np.array_equal(old["train_idx"], idx_train)
                or not np.array_equal(old["val_idx"], idx_val)
                or not np.array_equal(old["features"], matrix)
            ):
                raise ValueError("Arsip fusion lama berbeda: " + rep)
    else:
        temporary = path.with_suffix(".npz.tmp")
        with temporary.open("wb") as stream:
            np.savez_compressed(
                stream,
                features=matrix,
                y=y_all,
                indices=np.arange(50000),
                train_idx=idx_train,
                val_idx=idx_val,
                run_id=RUN_ID,
                representation=rep,
                split_hash=split_hash,
                model_hashes=np.array([model_hashes[v] for v in VARIANTS]),
                boundaries=np.array(BOUNDARIES),
                block_names=np.array(BLOCK_NAMES),
                preprocessing_version="v5-seven-groups",
            )
        os.replace(temporary, path)
    combined_hashes[rep] = sha256_file(path)
    print("Arsip:", path, "SHA", combined_hashes[rep][:16])


## 9. Kunci provenance pada manifest


In [ ]:
manifest = {
    "run_id": RUN_ID,
    "seed": SEED,
    "split_hash": split_hash,
    "model_paths": model_paths,
    "model_hashes": model_hashes,
    "feature_archive_hashes": archive_hashes,
    "combined_archive_hashes": combined_hashes,
    "manual_source": manual_source,
    "block_names": BLOCK_NAMES,
    "boundaries": BOUNDARIES,
    "feature_layer": "embedding",
    "preprocessing_version": "v5-seven-groups",
}
path = OUT / "laporan" / "combination_manifest.json"
if path.exists():
    if json.loads(path.read_text(encoding="utf-8")) != json.loads(json.dumps(manifest)):
        raise ValueError("Manifest fusion v5 lama berbeda")
else:
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    os.replace(temporary, path)
print("Manifest kombinasi:", path)
